### **Python Code Splitter**

In [9]:
from pprint import pp
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [6]:
python_code = '''
import argparse
import hashlib
from datetime import datetime

import apache_beam as beam
from apache_beam.io import parquetio
from apache_beam.options.pipeline_options import PipelineOptions

# -------------------------------------------------------------------
# BigQuery Bronze Schema
# -------------------------------------------------------------------
BQ_SCHEMA = """
trip_id:STRING,
VendorID:INTEGER,
lpep_pickup_datetime:TIMESTAMP,
lpep_dropoff_datetime:TIMESTAMP,
store_and_fwd_flag:STRING,
RatecodeID:FLOAT,
PULocationID:INTEGER,
DOLocationID:INTEGER,
passenger_count:FLOAT,
trip_distance:FLOAT,
fare_amount:FLOAT,
extra:FLOAT,
mta_tax:FLOAT,
tip_amount:FLOAT,
tolls_amount:FLOAT,
ehail_fee:FLOAT,
improvement_surcharge:FLOAT,
total_amount:FLOAT,
payment_type:FLOAT,
trip_type:FLOAT,
congestion_surcharge:FLOAT,
cbd_congestion_fee:FLOAT,
ingestion_ts:TIMESTAMP
"""

PARQUET_COLUMNS = [
    "VendorID",
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "store_and_fwd_flag",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "ehail_fee",
    "improvement_surcharge",
    "total_amount",
    "payment_type",
    "trip_type",
    "congestion_surcharge",
    "cbd_congestion_fee",
]

# -------------------------------------------------------------------
# Helpers (Bronze-safe)
# -------------------------------------------------------------------
def safe_int(value):
    try:
        return int(value)
    except Exception:
        return None

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None

def safe_datetime(value):
    try:
        if isinstance(value, datetime):
            return value
        return datetime.fromisoformat(str(value))
    except Exception:
        return None

# -------------------------------------------------------------------
# Record parsing (physical validity only)
# -------------------------------------------------------------------
def parse_record(record):
    pickup_ts = safe_datetime(record.get("lpep_pickup_datetime"))
    dropoff_ts = safe_datetime(record.get("lpep_dropoff_datetime"))

    if pickup_ts is None or dropoff_ts is None:
        return None
    if pickup_ts > dropoff_ts:
        return None

    return {
        "VendorID": safe_int(record.get("VendorID")),
        "lpep_pickup_datetime": pickup_ts.isoformat(),
        "lpep_dropoff_datetime": dropoff_ts.isoformat(),
        "store_and_fwd_flag": record.get("store_and_fwd_flag") or "Unknown",
        "RatecodeID": safe_float(record.get("RatecodeID")),
        "PULocationID": safe_int(record.get("PULocationID")),
        "DOLocationID": safe_int(record.get("DOLocationID")),
        "passenger_count": safe_float(record.get("passenger_count")),
        "trip_distance": safe_float(record.get("trip_distance")),
        "fare_amount": safe_float(record.get("fare_amount")),
        "extra": safe_float(record.get("extra")),
        "mta_tax": safe_float(record.get("mta_tax")),
        "tip_amount": safe_float(record.get("tip_amount")),
        "tolls_amount": safe_float(record.get("tolls_amount")),
        "ehail_fee": safe_float(record.get("ehail_fee")),
        "improvement_surcharge": safe_float(record.get("improvement_surcharge")),
        "total_amount": safe_float(record.get("total_amount")),
        "payment_type": safe_float(record.get("payment_type")),
        "trip_type": safe_float(record.get("trip_type")),
        "congestion_surcharge": safe_float(record.get("congestion_surcharge")),
        "cbd_congestion_fee": safe_float(record.get("cbd_congestion_fee")),
    }

# -------------------------------------------------------------------
# Trip ID (deterministic technical surrogate)
# -------------------------------------------------------------------
def generate_trip_id(record):
    key = (
        f"{record.get('VendorID')}-"
        f"{record.get('lpep_pickup_datetime')}-"
        f"{record.get('lpep_dropoff_datetime')}-"
        f"{record.get('PULocationID')}-"
        f"{record.get('DOLocationID')}"
    )
    record["trip_id"] = hashlib.md5(key.encode("utf-8")).hexdigest()
    return record

def add_ingestion_ts(record):
    record["ingestion_ts"] = datetime.utcnow().isoformat() + "Z"
    return record

# -------------------------------------------------------------------
# Args
# -------------------------------------------------------------------
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project", required=True)
    parser.add_argument("--region", default="us-central1")
    parser.add_argument("--input_parquet", required=True)
    parser.add_argument("--runner", required=True)
    parser.add_argument("--output_table", required=True)
    parser.add_argument("--temp_location", required=True)
    parser.add_argument("--staging_location", required=True)
    parser.add_argument("--job_name", default="nyc-green-taxi-bronze-load")
    return parser.parse_known_args()

# -------------------------------------------------------------------
# Main
# -------------------------------------------------------------------
if __name__ == "__main__":
    args, pipeline_args = parse_args()

    pipeline_options = PipelineOptions(
        pipeline_args,
        runner=args.runner,   # DirectRunner for local test
        project=args.project,
        region=args.region,
        job_name=args.job_name,
        temp_location=args.temp_location,
        staging_location=args.staging_location,
        service_account_email="dataflow-sa@glass-chemist-483110-u0.iam.gserviceaccount.com"
    )

    with beam.Pipeline(options=pipeline_options) as p:

        records = (
            p
            | "Read Parquet" >> parquetio.ReadFromParquet(
                file_pattern=args.input_parquet,
                columns=PARQUET_COLUMNS,
            )
            | "Parse Record" >> beam.Map(parse_record)
            | "Drop Invalid Records" >> beam.Filter(lambda x: x is not None)
            | "Generate Trip ID" >> beam.Map(generate_trip_id)
            | "Add Ingestion Timestamp" >> beam.Map(add_ingestion_ts)
        )

        records | "Write to BigQuery (Bronze)" >> beam.io.WriteToBigQuery(
            table=args.output_table,
            schema=BQ_SCHEMA,
            write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
            create_disposition=beam.io.BigQueryDisposition.CREATE_NEVER,
            custom_gcs_temp_location=args.temp_location,
        )
'''

In [10]:
print(python_code)


import argparse
import hashlib
from datetime import datetime

import apache_beam as beam
from apache_beam.io import parquetio
from apache_beam.options.pipeline_options import PipelineOptions

# -------------------------------------------------------------------
# BigQuery Bronze Schema
# -------------------------------------------------------------------
BQ_SCHEMA = """
trip_id:STRING,
VendorID:INTEGER,
lpep_pickup_datetime:TIMESTAMP,
lpep_dropoff_datetime:TIMESTAMP,
store_and_fwd_flag:STRING,
RatecodeID:FLOAT,
PULocationID:INTEGER,
DOLocationID:INTEGER,
passenger_count:FLOAT,
trip_distance:FLOAT,
fare_amount:FLOAT,
extra:FLOAT,
mta_tax:FLOAT,
tip_amount:FLOAT,
tolls_amount:FLOAT,
ehail_fee:FLOAT,
improvement_surcharge:FLOAT,
total_amount:FLOAT,
payment_type:FLOAT,
trip_type:FLOAT,
congestion_surcharge:FLOAT,
cbd_congestion_fee:FLOAT,
ingestion_ts:TIMESTAMP
"""

PARQUET_COLUMNS = [
    "VendorID",
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "store_and_fwd_flag",
    "

In [19]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=700,
    chunk_overlap=100
)

In [20]:
# split the code

code_chunks = python_splitter.split_text(python_code)

pp(code_chunks)

['import argparse\n'
 'import hashlib\n'
 'from datetime import datetime\n'
 '\n'
 'import apache_beam as beam\n'
 'from apache_beam.io import parquetio\n'
 'from apache_beam.options.pipeline_options import PipelineOptions',
 '# -------------------------------------------------------------------\n'
 '# BigQuery Bronze Schema\n'
 '# -------------------------------------------------------------------\n'
 'BQ_SCHEMA = """\n'
 'trip_id:STRING,\n'
 'VendorID:INTEGER,\n'
 'lpep_pickup_datetime:TIMESTAMP,\n'
 'lpep_dropoff_datetime:TIMESTAMP,\n'
 'store_and_fwd_flag:STRING,\n'
 'RatecodeID:FLOAT,\n'
 'PULocationID:INTEGER,\n'
 'DOLocationID:INTEGER,\n'
 'passenger_count:FLOAT,\n'
 'trip_distance:FLOAT,\n'
 'fare_amount:FLOAT,\n'
 'extra:FLOAT,\n'
 'mta_tax:FLOAT,\n'
 'tip_amount:FLOAT,\n'
 'tolls_amount:FLOAT,\n'
 'ehail_fee:FLOAT,\n'
 'improvement_surcharge:FLOAT,\n'
 'total_amount:FLOAT,\n'
 'payment_type:FLOAT,\n'
 'trip_type:FLOAT,\n'
 'congestion_surcharge:FLOAT,\n'
 'cbd_congestion_fee:

In [21]:
from termcolor import colored, COLORS
from random import choice
def display_chunks(chunks):
    colors_list = list(COLORS.keys())[2:8]
    print(f"Total Number of Chunks: {len(chunks)}")
    for num, chunk in enumerate(chunks, 1):
        print(f"Chunk {num}: Length {len(chunk)} chars")
        print(colored(text=chunk, color=choice(colors_list)), end="\n\n")

In [22]:
display_chunks(code_chunks)

Total Number of Chunks: 14
Chunk 1: Length 190 chars
import argparse
import hashlib
from datetime import datetime

import apache_beam as beam
from apache_beam.io import parquetio
from apache_beam.options.pipeline_options import PipelineOptions

Chunk 2: Length 681 chars
# -------------------------------------------------------------------
# BigQuery Bronze Schema
# -------------------------------------------------------------------
BQ_SCHEMA = """
trip_id:STRING,
VendorID:INTEGER,
lpep_pickup_datetime:TIMESTAMP,
lpep_dropoff_datetime:TIMESTAMP,
store_and_fwd_flag:STRING,
RatecodeID:FLOAT,
PULocationID:INTEGER,
DOLocationID:INTEGER,
passenger_count:FLOAT,
trip_distance:FLOAT,
fare_amount:FLOAT,
extra:FLOAT,
mta_tax:FLOAT,
tip_amount:FLOAT,
tolls_amount:FLOAT,
ehail_fee:FLOAT,
improvement_surcharge:FLOAT,
total_amount:FLOAT,
payment_type:FLOAT,
trip_type:FLOAT,
congestion_surcharge:FLOAT,
cbd_congestion_fee:FLOAT,
ingestion_ts:TIMESTAMP
"""

Chunk 3: Length 629 chars
PARQUET_COLUMNS = [


In [23]:
python_splitter.get_separators_for_language(Language.PYTHON)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']

In [24]:
python_splitter.get_separators_for_language(Language.MARKDOWN)

['\n#{1,6} ',
 '```\n',
 '\n\\*\\*\\*+\n',
 '\n---+\n',
 '\n___+\n',
 '\n\n',
 '\n',
 ' ',
 '']

### **JSON Splitter**

In [25]:
JSON_DATA = {
    "company": "AI Research Corp",
    "departments": [
        {
            "name": "Machine Learning",
            "team_size": 25,
            "projects": [
                {
                    "id": "ML001",
                    "title": "Computer Vision System",
                    "description": "Developing advanced image recognition using CNNs",
                    "status": "active",
                    "team_members": ["Alice", "Bob", "Charlie"]
                },
                {
                    "id": "ML002",
                    "title": "NLP Platform",
                    "description": "Building transformer-based language models",
                    "status": "active",
                    "team_members": ["David", "Eve"]
                }
            ]
        },
        {
            "name": "Data Engineering",
            "team_size": 15,
            "projects": [
                {
                    "id": "DE001",
                    "title": "Data Pipeline",
                    "description": "ETL pipeline for real-time data processing",
                    "status": "active"
                }
            ]
        }
    ],
    "technologies": {
        "frameworks": ["TensorFlow", "PyTorch", "scikit-learn"],
        "languages": ["Python", "R", "Julia"],
        "cloud": ["AWS", "Google Cloud", "Azure"]
    },
    "metadata": {
        "founded": 2020,
        "headquarters": "San Francisco",
        "employees": 150
    }
}

In [27]:
from langchain_text_splitters import RecursiveJsonSplitter

In [42]:
json_splitter = RecursiveJsonSplitter(
    max_chunk_size=100
)

In [43]:
# return dictionaries

chunks_dict = json_splitter.split_json(json_data=JSON_DATA)

pp(chunks_dict)

[{'company': 'AI Research Corp',
  'departments': [{'name': 'Machine Learning',
                   'team_size': 25,
                   'projects': [{'id': 'ML001',
                                 'title': 'Computer Vision System',
                                 'description': 'Developing advanced image '
                                                'recognition using CNNs',
                                 'status': 'active',
                                 'team_members': ['Alice', 'Bob', 'Charlie']},
                                {'id': 'ML002',
                                 'title': 'NLP Platform',
                                 'description': 'Building transformer-based '
                                                'language models',
                                 'status': 'active',
                                 'team_members': ['David', 'Eve']}]},
                  {'name': 'Data Engineering',
                   'team_size': 15,
                   'projects

In [44]:
display_chunks(chunks_dict)

Total Number of Chunks: 5
Chunk 1: Length 2 chars
{'company': 'AI Research Corp', 'departments': [{'name': 'Machine Learning', 'team_size': 25, 'projects': [{'id': 'ML001', 'title': 'Computer Vision System', 'description': 'Developing advanced image recognition using CNNs', 'status': 'active', 'team_members': ['Alice', 'Bob', 'Charlie']}, {'id': 'ML002', 'title': 'NLP Platform', 'description': 'Building transformer-based language models', 'status': 'active', 'team_members': ['David', 'Eve']}]}, {'name': 'Data Engineering', 'team_size': 15, 'projects': [{'id': 'DE001', 'title': 'Data Pipeline', 'description': 'ETL pipeline for real-time data processing', 'status': 'active'}]}]}

Chunk 2: Length 1 chars
{'technologies': {'frameworks': ['TensorFlow', 'PyTorch', 'scikit-learn']}}

Chunk 3: Length 1 chars
{'technologies': {'languages': ['Python', 'R', 'Julia']}}

Chunk 4: Length 1 chars
{'technologies': {'cloud': ['AWS', 'Google Cloud', 'Azure']}}

Chunk 5: Length 1 chars
{'metadata': {'fou

### **MARKDOWN Splitter**

In [48]:
MARKDOWN_TEXT = """# LangChain Tutorial Campux

A comprehensive, hands-on LangChain tutorial repository demonstrating how to work with various LangChain components, including different model types (LLMs, Chat Models, Embedding Models), prompt templates, output parsers, LangChain Expression Language (LCEL), and structured outputs.

## Overview

This repository is designed as a structured learning path for LangChain, moving from basic text generation to advanced techniques like chat memory management, conditional routing, semantic search, and structured data extraction. It integrates with major AI providers like OpenAI, Anthropic, Google Gemini, HuggingFace, and Ollama.

## Getting Started

### Prerequisites

1. Create and activate a Python environment (Python >= 3.10.13 recommended).
2. Install dependencies managed via `uv` or `pip`:

```bash
# Using pip
python -m pip install -r requirements.txt
```

3. Create a `.env` file in the root directory and add your API keys:

```env
OPENAI_API_KEY="your_openai_api_key"
ANTHROPIC_API_KEY="your_anthropic_api_key"
GOOGLE_API_KEY="your_google_api_key"
HUGGINGFACEHUB_API_TOKEN="your_hf_api_token"
```
*(Note: Not all keys are required; just the ones for the providers you intend to test).*

## Repository Structure & Content

### 1. `01_langchain_models/`
This module focuses on the core models provided by LangChain.

*   **`01_LLMs/`**: Working with standard text completion LLMs.
    *   `1_llm_demo.py`: Basic demonstration of invoking a standard LangChain LLM wrapper for text generation.
*   **`02_ChatModels/`**: Working with Chat Models that take a sequence of messages as input and return a message.
    *   `01_chatmodel_openai.py`: Chat model examples using OpenAI (`ChatOpenAI`).
    *   `02_chatmodel_anthropic.py`: Chat model examples using Anthropic's Claude (`ChatAnthropic`).
    *   `03_chatmodel_google.py`: Chat model examples using Google's Generative AI / Gemini (`ChatGoogleGenerativeAI`).
    *   `04_chatmodel_hugginface.py`: Connecting to open-source models hosted on Hugging Face (`ChatHuggingFace`).
    *   `05_chatmodel_ollama_local.py`: Running models locally using Ollama (`ChatOllama`).
*   **`03_EmbeddedModels/`**: Generating vector embeddings for text and documents.
    *   `01_embedding_openai_query.py`: Embedding a single query or text using OpenAI's embedding models.
    *   `02_embedding_openai_docs.py`: Embedding multiple documents in batch using OpenAI.
    *   `03_embedding_hf_local.py`: Generating embeddings locally without external API calls using HuggingFace sentence-transformers.
    *   `04_document_similarity.py`: A practical example of using embeddings to perform cosine similarity search between documents.

### 2. `02_langchain_prompts/`
This section covers how to manage and construct prompts, handle chat histories, and build interactive chatbots.

*   `chat_prompt_template.py`: Demonstrates the use of `ChatPromptTemplate` to format messages with system and human roles.
*   `messages.py`: Deep dive into LangChain message types (`SystemMessage`, `HumanMessage`, `AIMessage`).
*   `message_placeholder.py`: Using placeholders to dynamically inject chat histories into prompts.
*   `chatbot.py`: A functional chatbot implementation demonstrating memory and conversation flow.
*   `prompt_generator.py` & `prompt_ui.py`: Utilities and a simple UI (via Streamlit) for testing and generating dynamic prompts.
*   `template.json` & `chat_history.txt`: Resource files for prompt templates and persisting chat state.

### 3. `03_langchain_structure_output/`
Advanced examples of enforcing strict output schemas from LLMs, allowing you to get JSON or objects back reliably.

*   `pydantic_demo.py`: Basic introduction to defining data schemas using Pydantic.
*   `typeddict_demo.py`: Defining schemas using Python's native `TypedDict`.
*   `with_structured_output_pydantic.py`: Forcing an LLM to return data matching a Pydantic model (e.g., extracting entities from text).
*   `with_structured_output_typeddict.py`: Extracting structured data matching a `TypedDict` schema.
*   `with_structured_output_llama.py`: Example of structured output specifically tailored for Llama models.

### 4. `04_langchain_output_parsers/`
Examples of parsing raw text output from LLMs into structured formats via explicit parsers.

*   `stroutputparser.py` & `stroutputparser1.py`: Demonstrates the `StrOutputParser` to easily extract clean string content from complex `AIMessage` objects.
*   `pydanticoutputparser.py`: Using `PydanticOutputParser` to define schema and generate format instructions directly in the prompt.
*   `structuredoutputparser.py`: Using `StructuredOutputParser` and `ResponseSchema` for simpler structured data extraction.

### 5. `05_langchain_lcel/`
A deep dive into LangChain Expression Language (LCEL) for declarative chain composition.

*   `simple_chain.py`: Demonstrates the basic syntax (`prompt | llm | parser`).
*   `sequential_chain.py`: Chaining multiple prompts and models sequentially where the output of one feeds into the next.
*   `parallel_chain.py`: Utilizing `RunnableParallel` to execute multiple chains concurrently and combine their outputs.
*   `condiitonal_chain.py`: Demonstrates conditional routing using `RunnableBranch` and `RunnableLambda` based on sentiment analysis classification.

### Root Files
*   `main.py`: A simple entry script.
*   `pyproject.toml` / `requirements.txt`: Project dependencies including LangChain core, provider SDKs, Pydantic, Streamlit, and ML libraries (PyTorch, Transformers, Scikit-learn).
*   `uv.lock`: Lockfile for reproducible builds if using `uv`.

## Recommended Workflow

1.  **Start with Models**: Run examples in `01_langchain_models/` to understand basic invocation.
2.  **Learn Prompting**: Check out `02_langchain_prompts/` to learn how to construct complex prompts.
3.  **Explore LCEL**: Dive into `05_langchain_lcel/` to understand how chains are built using the pipe syntax.
4.  **Try Parsers**: Run scripts in `04_langchain_output_parsers/` to extract cleanly formatted responses.
5.  **Master Structured Output**: Explore `03_langchain_structure_output/` to see advanced techniques for returning exact schemas.

## License

This repository is intended for educational, tutorial, and experimentation purposes.
"""

In [49]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [50]:
headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3")
]


In [57]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

In [58]:
# split the text

markdown_chunks = markdown_splitter.split_text(MARKDOWN_TEXT)

pp(markdown_chunks)

[Document(metadata={'Header_1': 'LangChain Tutorial Campux'}, page_content='# LangChain Tutorial Campux  \nA comprehensive, hands-on LangChain tutorial repository demonstrating how to work with various LangChain components, including different model types (LLMs, Chat Models, Embedding Models), prompt templates, output parsers, LangChain Expression Language (LCEL), and structured outputs.'),
 Document(metadata={'Header_1': 'LangChain Tutorial Campux', 'Header_2': 'Overview'}, page_content='## Overview  \nThis repository is designed as a structured learning path for LangChain, moving from basic text generation to advanced techniques like chat memory management, conditional routing, semantic search, and structured data extraction. It integrates with major AI providers like OpenAI, Anthropic, Google Gemini, HuggingFace, and Ollama.'),
 Document(metadata={'Header_1': 'LangChain Tutorial Campux', 'Header_2': 'Getting Started', 'Header_3': 'Prerequisites'}, page_content='## Getting Started  \

In [59]:
for doc in markdown_chunks:
    print(doc.page_content, end="\n\n")

# LangChain Tutorial Campux  
A comprehensive, hands-on LangChain tutorial repository demonstrating how to work with various LangChain components, including different model types (LLMs, Chat Models, Embedding Models), prompt templates, output parsers, LangChain Expression Language (LCEL), and structured outputs.

## Overview  
This repository is designed as a structured learning path for LangChain, moving from basic text generation to advanced techniques like chat memory management, conditional routing, semantic search, and structured data extraction. It integrates with major AI providers like OpenAI, Anthropic, Google Gemini, HuggingFace, and Ollama.

## Getting Started  
### Prerequisites  
1. Create and activate a Python environment (Python >= 3.10.13 recommended).
2. Install dependencies managed via `uv` or `pip`:  
```bash
# Using pip
python -m pip install -r requirements.txt
```  
3. Create a `.env` file in the root directory and add your API keys:  
```env
OPENAI_API_KEY="your_o